# Обучение SegFormer для задачи сегментации повреждений фасада

Ноутбук предназначен для платформы Kaggle. Предварительно необходимо загрузить датасет (картинки + маски из CVAT) в качестве Kaggle Dataset (например, `facade-damage-segmentation`).

In [1]:
!pip install -q transformers evaluate datasets albumentations accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.0 MB/s eta 0:00:00


In [2]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
import evaluate
import albumentations as A
from torch.utils.data import Dataset, DataLoader
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor, TrainingArguments, Trainer
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используем устройство: {device}")

Используем устройство: cpu


## 1. Настройка путей и классов

Укажи правильные пути к твоему датасету в Kaggle.

In [3]:
# ПУТИ В KAGGLE (Обычно /kaggle/input/твое-имя-датасета/)
DATASET_PATH = '/kaggle/input/facade-damage-segmentation'  # <-- ЗАМЕНИТЕ НА ВАШ ПУТЬ В KAGGLE
IMAGES_DIR = os.path.join(DATASET_PATH, 'images')          # Папка с исходными .jpg
MASKS_DIR = os.path.join(DATASET_PATH, 'masks')            # Папка с экспортированными из CVAT масками (RGB)

# Маппинг классов
CLASS_MAPPING = {
    0: "background",
    1: "Bio-chemical",
    2: "Antropological",
    3: "Mechanical",
    255: "Ignored" 
}
id2label = CLASS_MAPPING
label2id = {v: k for k, v in id2label.items()}

cvat_colors_rgb = {
    "background": [0, 0, 0],
    "Bio-chemical": [38, 221, 98],
    "Antropological": [222, 28, 123],
    "Mechanical": [54, 21, 217],
    "Ignored": [221, 255, 51]
}

## 2. Подготовка Данных (Конвертация RGB-масок в Индексные)

In [4]:
# Чтобы не конвертировать маски каждый раз с нуля, мы сделаем это один раз и сохраним в рабочую директорию Kaggle (/kaggle/working)
PROCESSED_MASKS_DIR = '/kaggle/working/masks_indexed'
os.makedirs(PROCESSED_MASKS_DIR, exist_ok=True)

def rgb_mask_to_index_mask(rgb_mask_bgr, color_map):
    index_mask = np.zeros(rgb_mask_bgr.shape[:2], dtype=np.uint8)
    class_to_idx = {name: idx for idx, name in CLASS_MAPPING.items()} 
    
    for class_name, class_idx in class_to_idx.items():
        rgb = color_map[class_name]
        bgr = [rgb[2], rgb[1], rgb[0]]
        
        lower = np.array(bgr, dtype=np.uint8)
        upper = np.array(bgr, dtype=np.uint8)
        class_pixels = cv2.inRange(rgb_mask_bgr, lower, upper)
        index_mask[class_pixels == 255] = class_idx
        
    return index_mask

# Собираем пары (image, mask)
from glob import glob
mask_files = glob(os.path.join(MASKS_DIR, "*.png"))

dataset_pairs = []

print("Конвертация масок и сбор пар...")
for mask_path in tqdm(mask_files, total=len(mask_files)):
    filename = os.path.basename(mask_path)
    
    # Ищем исходную картинку
    image_path_png = os.path.join(IMAGES_DIR, filename)
    image_path_jpg = os.path.join(IMAGES_DIR, os.path.splitext(filename)[0] + '.jpg')
    
    img_path = None
    if os.path.exists(image_path_png):
        img_path = image_path_png
    elif os.path.exists(image_path_jpg):
        img_path = image_path_jpg
        
    if img_path is None:
        continue
        
    # Читаем RGB маску и конвертируем
    rgb_mask_bgr = cv2.imread(mask_path, cv2.IMREAD_COLOR)
    if rgb_mask_bgr is None: continue
        
    index_mask = rgb_mask_to_index_mask(rgb_mask_bgr, cvat_colors_rgb)
    
    # Сохраняем
    indexed_mask_path = os.path.join(PROCESSED_MASKS_DIR, filename)
    cv2.imwrite(indexed_mask_path, index_mask)
    
    dataset_pairs.append({
        "image": img_path,
        "mask": indexed_mask_path
    })

print(f"Собрано {len(dataset_pairs)} пар картинок и масок.")

Конвертация масок и сбор пар...


0it [00:00, ?it/s]

Собрано 0 пар картинок и масок.


## 3. Train/Val Split

Разделим на обучение (80%) и валидацию (20%)

In [5]:
train_pairs, val_pairs = train_test_split(dataset_pairs, test_size=0.2, random_state=42)
print(f"Обучающая выборка: {len(train_pairs)}")
print(f"Валидационная выборка: {len(val_pairs)}")

ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

## 4. PyTorch Dataset & Augmentations

In [ ]:
class DamageSegmentationDataset(Dataset):
    def __init__(self, dataset_pairs, transform=None):
        self.dataset_pairs = dataset_pairs
        self.transform = transform
        # ImageProcessor нормирует картинки под SegFormer
        self.processor = SegformerImageProcessor.from_pretrained("nvidia/mit-b2")

    def __len__(self):
        return len(self.dataset_pairs)

    def __getitem__(self, idx):
        pair = self.dataset_pairs[idx]
        
        # Считываем изображение (RGB для Albumentations)
        image = cv2.imread(pair["image"])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Считываем индексную маску
        mask = cv2.imread(pair["mask"], cv2.IMREAD_GRAYSCALE)
        
        # Применяем аугментации (если есть)
        if self.transform is not None:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        # Подготовка под HuggingFace Segformer Processor
        # Он занимается ресайзом и нормализацией (ImageNet norm)
        # HuggingFace ожидает PIL картинку
        inputs = self.processor(images=Image.fromarray(image), segmentation_maps=mask, return_tensors="pt")
        
        # Убираем лишнюю размерность batch, созданную процессором
        inputs = {k: v.squeeze() for k, v in inputs.items()}
        
        return inputs

# Аугментации на тренировку (чуть добавляем вариативности)
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
])

# На валидацию аугментации не нужны
val_transform = None

train_dataset = DamageSegmentationDataset(train_pairs, transform=train_transform)
val_dataset = DamageSegmentationDataset(val_pairs, transform=val_transform)

## 5. Вычисление Весов Классов (Loss Weights)

У нас сильный дисбаланс, поэтому мы вычислим веса для CrossEntropyLoss, чтобы штрафовать редкие классы сильнее.

In [ ]:
# Твои цифры баланса (примерные доли)
class_fractions = {
    0: 0.74,     # background
    1: 0.1047,   # Bio-chemical
    2: 0.0076,   # Antropological
    3: 0.1337,   # Mechanical
    # 255 (Ignored) не взвешивается, он игнорируется лоссом
}

# Обратная пропорция
weights = []
for i in range(4): # 4 активных класса: от 0 до 3
    weights.append(1.0 / class_fractions[i])

# Нормализуем так, чтобы минимальный вес (у фона) был около 1.0
weights = np.array(weights)
weights = weights / weights.min()

class_weights_tensor = torch.tensor(weights, dtype=torch.float).to(device)
print("Веса классов для CrossEntropyLoss:")
for i, w in enumerate(weights):
    print(f"Класс {id2label[i]}: {w:.4f}")

## 6. Инициализация Модели и Метрик

Используем кастомный класс `WeightedTrainer`, чтобы передать наши веса в функцию потерь.

In [ ]:
from torch import nn

# Кастомный Trainer с весами
class ClassWeightsTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Скейл масок к размеру логитов (1/4 от оригинального изображения)
        upsampled_logits = nn.functional.interpolate(
            logits, 
            size=labels.shape[-2:], 
            mode="bilinear", 
            align_corners=False
        )
        
        # ignore_index=255 значит, что лосс не считается на пикселях класса Ignored!
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor, ignore_index=255)
        loss = loss_fct(upsampled_logits, labels)
        
        return (loss, outputs) if return_outputs else loss


# Загружаем метрику mIoU
metric = evaluate.load("mean_iou")

def compute_metrics(eval_pred):
    with torch.no_grad():
        logits, labels = eval_pred
        logits_tensor = torch.from_numpy(logits)
        # Скейл логитов до оригинального размера для честного расчета масок
        logits_tensor = nn.functional.interpolate(
            logits_tensor,
            size=labels.shape[-2:],
            mode="bilinear",
            align_corners=False,
        ).argmax(dim=1)
        
        pred_labels = logits_tensor.detach().cpu().numpy()
        
        metrics = metric.compute(
            predictions=pred_labels,
            references=labels,
            num_labels=len(id2label) - 1, # Не считаем Ignored
            ignore_index=255,
            reduce_labels=False,
        )
        
        # Сохраняем IoU по каждому классу
        per_category_iou = metrics.pop("per_category_iou").tolist()
        for i, val in enumerate(per_category_iou):
            if i < len(id2label) - 1:
                metrics[f"iou_{id2label[i]}"] = val
            
        return metrics

# Загрузка модели (Берем ImageNet encoder B2)
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b2",
    num_labels=len(id2label) - 1, # 4 рабочих класса (0-3)
    id2label={k: v for k,v in id2label.items() if k != 255},
    label2id={v: k for k,v in id2label.items() if k != 255},
    ignore_mismatched_sizes=True # Важно, т.к. мы меняем голову под свои 4 класса
)

## 7. Параметры обучения

Рекомендую поставить 30-50 эпох, так как датасет небольшой. С оптимизатором AdamW и Cosine расписанием SegFormer учится быстро.

In [ ]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/segformer-b2-facade-damage",
    learning_rate=6e-5,
    num_train_epochs=50,
    per_device_train_batch_size=8, # Если не влезает в GPU (OOM), уменьши до 4 или 2
    per_device_eval_batch_size=8,
    save_total_limit=3,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    eval_steps=10,
    remove_unused_columns=False,
    push_to_hub=False,
    load_best_model_at_end=True,
    metric_for_best_model="mean_iou",
)

trainer = ClassWeightsTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

## 8. Запуск Обучения 🚀

In [ ]:
print("Погнали учиться!")
trainer.train()

ПОСЛЕ ОБУЧЕНИЯ:
Чтобы не потерять модель, она сохранится в папке `/kaggle/working/segformer-b2-facade-damage`.
Скачай её или отправь в Google Drive. Загрузи веса (`checkpoint-XXX`) на свой ПК, чтобы потом использовать в диссертации (Фаза 4).